# 03 · Analysis nodes

Block 3 check: the tune nodes (stage z, C1, A1) and the segment node, each run alone on a hand-made state, then overview → segment → EDS by hand.

**On the twin the tuning cannot find a real optimum**: the twin's image does not change with z, C1 or A1, so scores move only with noise and the search ends at a random point inside its limits. That is expected. Here we check the mechanics — limits, moves, corrector steps, restore on error, files. That the nodes *find* an optimum is tested in `tests/test_nodes_analysis.py` with a fake microscope whose image blurs.

**Before running (twin):** `uv run python scripts/twin.py start`.

In [ ]:
TARGET = "twin"   # "twin" or "real"

SETTINGS = {
    "target": TARGET,
    "target_xy_m": (5e-9, 0.0),
    "tuning_fov_m": 40e-9,
    "tuning": {"z": {"max_iterations": 8}},   # defaults: config.TUNING (z: 20 iterations)
    "run_tag": "block3",
}

In [ ]:
import json
from pathlib import Path

import numpy as np
from IPython.display import Image, display

from stem_agent.config import search_settings
from stem_agent.mcp import session
from stem_agent.nodes import common
from stem_agent.nodes.eds_node import eds_node
from stem_agent.nodes.overview_node import overview_node
from stem_agent.nodes.preflight_node import preflight_node
from stem_agent.nodes.segment_node import segment_node
from stem_agent.nodes.set_fov_node import set_fov_node
from stem_agent.nodes.status_node import status_end_node, status_start_node
from stem_agent.nodes.tune_node import focus_z_node, tune_a1_node, tune_c1_node


def apply(state, update):
    """Merge a node's update into a new state dict."""
    print("->", sorted(update), "| run_status:", update.get("run_status", state.get("run_status")))
    if "error" in update:
        print("ERROR:", update["error"])
    if "tuning" in update:
        update = {**update, "tuning": {**state.get("tuning", {}), **update["tuning"]}}
    return {**state, **update}


async def twin_coefficients():
    """The twin's internal aberrations (twin only: the real corrector has no such readback)."""
    if TARGET != "twin":
        return None
    return json.loads(await session.call_tool("DigitalTwinCorrector_get_aberrations_coeff_sim"))


state = {"settings": SETTINGS}

## 1 · Preflight and status
Preflight also checks the tuning settings: allowed metric, `step <= max_offset`, `max_offset` within half the per-move limit.

In [ ]:
state = apply(state, await preflight_node(state))
state = apply(state, await status_start_node(state))
for name in ("z", "C1", "A1"):
    print(name, search_settings(SETTINGS, name))

## 2 · Focus (stage z)
Expected: every z within ±`max_offset` of the start, the stage ends at the best point, `tune_z.json` + `tune_z.png` saved. On the twin the scores are nearly flat.

In [ ]:
z_start = state["current"]["stage"][2]
state = apply(state, await focus_z_node(state))
result = state["tuning"]["z"]
points = json.loads(Path(result["history_path"]).read_text())["points"]
furthest = max(abs(p["reached_m"][0]) for p in points)
print(f"z start {z_start * 1e6:+.3f} um -> now {state['current']['stage'][2] * 1e6:+.3f} um | "
      f"furthest {furthest * 1e6:.2f} um (limit {result['settings']['max_offset_m'] * 1e6:.0f}) | "
      f"stop: {result['stop_reason']}")
Image(result["figure_path"])

## 3 · Set FOV, then C1
The twin's internal C1 is read before and after. Its change divided by the node's total shows the twin's **sign convention** (block 3 check: −1, the twin subtracts what it is sent).

In [ ]:
state = apply(state, await set_fov_node(state))
before = await twin_coefficients()
state = apply(state, await tune_c1_node(state))
after = await twin_coefficients()
total = state["current"]["c1_offset_m"]
print(f"node total C1 {total * 1e9:+.3f} nm")
if before:
    change = after["C1"][0] - before["C1"][0]
    print(f"twin C1 {before['C1'][0] * 1e9:+.3f} -> {after['C1'][0] * 1e9:+.3f} nm | "
          f"change / node total = {change / total if total else float('nan'):+.3f}")
Image(state["tuning"]["C1"]["figure_path"])

## 4 · A1
Same check in 2D.

In [ ]:
before = await twin_coefficients()
state = apply(state, await tune_a1_node(state))
after = await twin_coefficients()
total = np.array(state["current"]["a1_offset_m"])
print(f"node total A1 {np.round(total * 1e9, 3).tolist()} nm")
if before:
    change = np.array(after["A1"]) - np.array(before["A1"])
    print(f"twin A1 change {np.round(change * 1e9, 3).tolist()} nm | change / node total = "
          f"{np.round(np.divide(change, total, out=np.full(2, np.nan), where=total != 0), 3).tolist()}")
Image(state["tuning"]["A1"]["figure_path"])

## 5 · Forced error: restore
Test only: the 5th image of a search fails (a wrapper around `common.acquire_haadf`). Expected: `run_status: failed`, and z / C1 back exactly where that node started.

In [ ]:
original_acquire = common.acquire_haadf


def fail_after(n):
    count = {"images": 0}
    async def acquire(detector="haadf"):
        count["images"] += 1
        if count["images"] > n:
            raise RuntimeError("test only: injected acquisition failure")
        return await original_acquire(detector)
    return acquire


try:
    z_before = (await common.get_stage())[2]
    common.acquire_haadf = fail_after(4)
    update = await focus_z_node(state)
    print(update["run_status"], "| z restored:", (await common.get_stage())[2] == z_before)

    c1_before = await twin_coefficients()
    common.acquire_haadf = fail_after(4)
    update = await tune_c1_node(state)
    c1_after = await twin_coefficients()
    print(update["run_status"], "| C1 restored:",
          None if c1_before is None else abs(c1_after["C1"][0] - c1_before["C1"][0]) < 1e-15)
finally:
    common.acquire_haadf = original_acquire

## 6 · Overview → segment → EDS
By hand, the last part of the workflow. Expected: particles outlined with aim points inside; on the twin, EDS at the aim points gives particle compositions (max fraction ≳ 0.3).

In [ ]:
state = apply(state, await overview_node(state))
state = apply(state, await segment_node(state))
display(Image(state["segmentation"]["overlay_path"]))

state = apply(state, await eds_node(state))
hits = [bool(np.load(r["npy_path"]).max() > 0.25) for r in state["spectra"]]
print(f"EDS on {len(hits)} particles: {sum(hits)} look like a particle composition")
Image(Path(state["run"]["run_dir"]) / "eds.png")

## 7 · Status end

In [ ]:
state = apply(state, await status_end_node(state))
run_dir = Path(state["run"]["run_dir"])
sorted(str(p.relative_to(run_dir)) for p in run_dir.rglob("*") if p.is_file())